# Exp9.0 — Within-user unseen-segment generalization

Aggregation-only notebook for `within_user_segment_generalization_v1`.
It reads finalized CSV/JSON artifacts; it does not train models or refit classifiers.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
ART = REPO_ROOT / "notebooks" / "artifacts" / "experiment_9_0_within_user_generalization" / "within_user_segment_generalization_v1"
ART


## Dataset audit

The prepare stage enforces source-trial isolation and writes the insufficient-pair audit before any model job starts.


In [ ]:
audit = json.loads((ART / "manifests" / "audit.json").read_text())
audit


In [ ]:
initial = pd.read_csv(ART / "manifests" / "initial_pair_summary.csv")
insufficient = pd.read_csv(ART / "manifests" / "insufficient_user_class_pairs.csv")
print("user-class pairs:", len(initial))
print("insufficient pairs:", len(insufficient))
insufficient.head(20)


## Aggregate train / validation / test metrics


In [ ]:
metric_summary = pd.read_csv(ART / "metric_summary.csv")
metric_summary


In [ ]:
metric_runs = pd.read_csv(ART / "metric_runs.csv")
fig, ax = plt.subplots(figsize=(8, 4.5))
for method, frame in metric_runs.groupby("method"):
    means = frame.groupby("split")["balanced_accuracy"].mean().reindex(["train", "val", "test"])
    ax.plot(means.index, means.values, marker="o", label=method)
ax.set_ylabel("Balanced accuracy")
ax.set_title("Exp9.0 within-user generalization")
ax.legend()
fig.tight_layout()


## Per-user variability


In [ ]:
per_user = pd.read_csv(ART / "per_user_summary.csv")
test_users = per_user[per_user["split"] == "test"].copy()
test_users[["method", "user", "balanced_accuracy_mean", "balanced_accuracy_std", "macro_f1_mean"]].sort_values(["method", "balanced_accuracy_mean"])


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for method, frame in test_users.groupby("method"):
    ordered = frame.sort_values("user")
    ax.plot(ordered["user"], ordered["balanced_accuracy_mean"], marker="o", label=method)
ax.set_ylabel("Mean test balanced accuracy")
ax.set_xlabel("User")
ax.tick_params(axis="x", rotation=90)
ax.set_title("Within-user unseen-segment BA by user")
ax.legend()
fig.tight_layout()


## Per-class behavior


In [ ]:
per_class = pd.read_csv(ART / "per_class_summary.csv")
per_class[per_class["split"] == "test"][["method", "class_label", "recall_mean", "f1_mean", "support_mean"]].sort_values(["method", "f1_mean"])


## Within-user versus historical cross-user reference


In [ ]:
comparison = pd.read_csv(ART / "within_vs_cross_user.csv")
comparison


In [ ]:
if "cross_user_test_ba" in comparison.columns:
    plot = comparison.dropna(subset=["within_user_test_ba_mean", "cross_user_test_ba"])
    if not plot.empty:
        x = range(len(plot))
        fig, ax = plt.subplots(figsize=(7, 4.5))
        width = 0.35
        ax.bar([i - width/2 for i in x], plot["within_user_test_ba_mean"], width=width, label="within-user")
        ax.bar([i + width/2 for i in x], plot["cross_user_test_ba"], width=width, label="cross-user")
        ax.set_xticks(list(x), plot["method"])
        ax.set_ylabel("Balanced accuracy")
        ax.set_title("Within-user vs cross-user test BA")
        ax.legend()
        fig.tight_layout()


## Mean test confusion matrices across split seeds


In [ ]:
conf = pd.read_csv(ART / "confusion_summary.csv")
for method, frame in conf[conf["split"] == "test"].groupby("method"):
    matrix = frame.pivot(index="true_label", columns="pred_label", values="mean_count")
    matrix = matrix.reindex(index=sorted(matrix.index), columns=sorted(matrix.columns))
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix.to_numpy(), aspect="auto")
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=90)
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{method}: mean test confusion")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
